In [ ]:
from utils import preprocess
from utils import models

df_customers, df_products, df_transactions = preprocess.load_complete_dataset_filtered_number_customers(5000)



In [ ]:
import pandas as pd

df_transactions['t_dat'] = pd.to_datetime(df_transactions['t_dat'])

# Leave-one-out: el test es el último artículo comprado por cada usuario
last_purchase_idx = df_transactions.groupby('customer_id')['t_dat'].idxmax()
df_test  = df_transactions.loc[last_purchase_idx]
df_train = df_transactions.drop(index=last_purchase_idx)

# Solo usuarios con historial en train (al menos 1 compra previa)
users_with_train = set(df_train['customer_id'].unique())
eval_users = [u for u in df_test['customer_id'].unique() if u in users_with_train]

ground_truth = df_test.set_index('customer_id')['article_id'].to_dict()
actual = [[ground_truth[u]] for u in eval_users]

print(f"Usuarios para evaluación: {len(eval_users)}")
print(f"Transacciones train: {len(df_train):,} | test: {len(df_test):,}")

print(f"Usuarios para evaluación: {len(eval_users)}")
print(f"Transacciones train: {len(df_train)} | test: {len(df_test)}")

# Ground truth: artículos comprados en la semana de test por cada usuario
ground_truth = (
    df_test[df_test['customer_id'].isin(eval_users)]
    .groupby('customer_id')['article_id']
    .apply(list)
    .to_dict()
)
actual = [ground_truth[u] for u in eval_users]

Usuarios para evaluación: 449
Transacciones train: 10,827 | test: 500
Usuarios para evaluación: 449
Transacciones train: 10827 | test: 500


In [ ]:
from utils import models

K = 20

# --- Baseline: aleatorio ---
pred_random = models.predict_random(df_train, eval_users, k=K)

# --- Baseline: popularidad ---
pred_popular = models.predict_popular(df_train, eval_users, k=K)

# --- Modelo: clustering ---
pred_cluster, _, _, _ = models.predict_cluster(
    df_train, df_customers, df_products,
    customer_ids=eval_users, K=K, top_n=K, explore_k=False,
)

# --- Evaluación MAP@12 ---
results = {
    'Random':    models.mapk(actual, pred_random,  k=K),
    'Popular':   models.mapk(actual, pred_popular, k=K),
    'Cluster':   models.mapk(actual, pred_cluster, k=K),
}

print("MAP@12:")
for name, score in sorted(results.items(), key=lambda x: -x[1]):
    print(f"  {name:<10} {score:.4f}")

MAP@12:
  Cluster    0.0046
  Popular    0.0007
  Random     0.0001


In [ ]:
items_per_user = [len(v) for v in ground_truth.values()]
print(f"Usuarios en evaluación:              {len(eval_users)}")
print(f"Media de artículos test por usuario: {sum(items_per_user)/len(items_per_user):.2f}")
print(f"Usuarios con ≥1 artículo en test:    {sum(1 for x in items_per_user if x >= 1)}")
print(f"Artículos únicos en catálogo:        {df_products['article_id'].nunique():,}")

Usuarios en evaluación:              4542
Media de artículos test por usuario: 1.00
Usuarios con ≥1 artículo en test:    4542
Artículos únicos en catálogo:        105,542
